# Persistent entropy using absolute correlation distance

This notebook reproduces the persistent-entropy analysis in which the distance matrix is defined as `1 - abs(correlation)`. It keeps the original GUDHI/Rips pipeline and the later Wilcoxon, rank-biserial correlation and bootstrap confidence-interval summary.


## Data loading


In [1]:
# Private matrix-data configuration and paired loading
from pathlib import Path
import re
import glob
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
CANDIDATE_DATA_DIRS = [
    PROJECT_DIR / "data_private" / "connectivity_matrices",
    PROJECT_DIR / "public_repository" / "data_private" / "connectivity_matrices",
    PROJECT_DIR,
    PROJECT_DIR.parent,
]

# In the public repository, place private .txt matrices under:
#   data_private\connectivity_matrices\
# The additional candidates support the local preparation workspace only.
DATA_DIR = next((path for path in CANDIDATE_DATA_DIRS if any(path.glob("*before*.txt"))), CANDIDATE_DATA_DIRS[0])

BEFORE_PATTERN = "*before*.txt"
AFTER_PATTERN = "*after*.txt"


def _subject_id(path):
    name = Path(path).name.lower()
    match = re.search(r"suj\s*[_-]?(\d+)|s[_-]?(\d+)", name)
    if match:
        return int(match.group(1) or match.group(2))
    return name


def _load_paired_connectivity_matrices(data_dir):
    before_files = sorted(Path(data_dir).glob(BEFORE_PATTERN), key=_subject_id)
    after_files = sorted(Path(data_dir).glob(AFTER_PATTERN), key=_subject_id)

    if len(before_files) == 0 or len(after_files) == 0:
        raise FileNotFoundError(
            f"No paired .txt connectivity matrices were found in {Path(data_dir).resolve()}. "
            "Set DATA_DIR to the private matrix folder."
        )

    before_ids = [_subject_id(path) for path in before_files]
    after_ids = [_subject_id(path) for path in after_files]
    if before_ids != after_ids:
        raise ValueError(
            "Before/after subject identifiers do not match.\n"
            f"before={before_ids}\nafter={after_ids}"
        )

    before_array = np.array([np.loadtxt(path) for path in before_files])
    after_array = np.array([np.loadtxt(path) for path in after_files])

    if before_array.shape != after_array.shape:
        raise ValueError(f"Shape mismatch: before={before_array.shape}, after={after_array.shape}")

    return before_array, after_array, before_files, after_files, before_ids


before, after, txt_files_before, txt_files_after, subject_ids = _load_paired_connectivity_matrices(DATA_DIR)

print(f"Data directory: {Path(DATA_DIR).resolve()}")
print(f"Paired subjects: {subject_ids}")
print(f"Number of before files: {len(txt_files_before)}")
print(f"Number of after files : {len(txt_files_after)}")
print(f"before shape: {before.shape}")
print(f"after shape : {after.shape}")


Data directory: D:\OneDrive\Coding\article_repository
Paired subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Number of before files: 9
Number of after files : 9
before shape: (9, 104, 104)
after shape : (9, 104, 104)


## Persistent entropy in dimensions 0 to 3


In [2]:
# Cálculo entropia persistente

import gudhi
import scipy
import numpy as np
import matplotlib.pyplot as plt
from gudhi import representations
from tqdm import tqdm  # Importação da biblioteca de barra de progresso

# Alteração: Adicionado o parametro 'dim'
def c_pers(matrix, dim):
    dist_matrix = 1 - np.abs(matrix)
    
    # Obs: O .tolist() é importante para garantir compatibilidade, caso matrix seja numpy
    if isinstance(dist_matrix, np.ndarray):
        dist_matrix_list = dist_matrix.tolist()
    else:
        dist_matrix_list = dist_matrix

    scomplex = gudhi.RipsComplex(distance_matrix=dist_matrix_list, max_edge_length=0.99)

    # Alteração: Aumentado para 4 para permitir o cálculo da dimensão 3
    s_tree = scomplex.create_simplex_tree(max_dimension=4)

    a = s_tree.persistence(homology_coeff_field=2, min_persistence=0)
    
    # Alteração: Usa a dimensão passada como argumento
    b = s_tree.persistence_intervals_in_dimension(dim)
    
    gudhi.plot_persistence_barcode(a, legend=True)
    
    # Alteração: Desabilitado show e adicionado close para não acumular memória
    # plt.show()
    plt.close()

    # Lógica original mantida
    soma = 0
    for i in range(0, len(b)):
        if b[i][1] < float('inf'):
            soma += b[i][1]

    entropy = 0
    # Adicionei verificação simples para evitar divisão por zero se a soma for 0
    if soma > 0:
        for i in range(0, len(b)):
            if b[i][1] < float('inf'):        
                entropy += -(b[i][1]/soma)*np.log(b[i][1]/soma)
    
    return entropy

antes = []
depois = []

# Verificação se as listas existem e têm o mesmo tamanho
if 'before' in locals() and 'after' in locals() and len(before) == len(after):
    
    # --- ALTERAÇÃO AQUI: tqdm envolve o range para criar a barra ---
    # desc="Processando" adiciona um texto antes da barra
    for i in tqdm(range(0, len(before)), desc="Processando Sujeitos"):
        
        # Vetores temporários para este sujeito
        sujeito_antes_dims = []
        sujeito_depois_dims = []
        
        # Loop externo solicitando as dimensões 0, 1, 2 e 3
        for d in range(4):
            sujeito_antes_dims.append(c_pers(before[i], d))
            sujeito_depois_dims.append(c_pers(after[i], d))
        
        # Adiciona o vetor [d0, d1, d2, d3] à lista principal
        antes.append(sujeito_antes_dims)
        depois.append(sujeito_depois_dims)
else:
    print("Erro: Variáveis 'before'/'after' não definidas ou com tamanhos diferentes.")

print("\nProcessamento concluído.")
# print(antes) # Comentado pois pode ser muito grande para exibir
# print(depois)

# Exemplo de como chamar a estatística para a dimensão 0, se necessário:
# c_estat([x[0] for x in antes], [y[0] for y in depois], 0)


C:\ProgramData\anaconda3\Lib\site-packages\gudhi\persistence_graphical_tools.py:129: UserWarning: usetex mode requires TeX.
  warnings.warn("usetex mode requires TeX.")
Processando Sujeitos: 100%|██████████| 9/9 [12:27<00:00, 83.03s/it] 


Processamento concluído.


## Paired statistical analysis


In [3]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata, wilcoxon, bootstrap

# ============================================================
# ESTA CÉLULA ASSUME QUE "antes" E "depois" JÁ ESTÃO NA MEMÓRIA
# no formato (n_sujeitos, n_dimensoes)
# ============================================================

antes_arr = np.asarray(antes, dtype=float)
depois_arr = np.asarray(depois, dtype=float)

if antes_arr.shape != depois_arr.shape:
    raise ValueError(
        f"'antes' e 'depois' precisam ter a mesma shape. "
        f"Recebi {antes_arr.shape} e {depois_arr.shape}."
    )

if antes_arr.ndim != 2:
    raise ValueError(
        f"Esperado array 2D (sujeitos x dimensões). "
        f"Recebi ndim = {antes_arr.ndim}."
    )

n_sujeitos, n_dimensoes = antes_arr.shape

print("=" * 90)
print("INÍCIO DA ANÁLISE ESTATÍSTICA")
print("=" * 90)
print(f"Número de sujeitos   : {n_sujeitos}")
print(f"Número de dimensões  : {n_dimensoes}")
print(f"Shape de 'antes'     : {antes_arr.shape}")
print(f"Shape de 'depois'    : {depois_arr.shape}")
print("\nObservações:")
print("- Diferenças calculadas como: antes - depois")
print("- RBC calculado da mesma forma que você já vinha usando")
print("- IC 95% calculado por bootstrap BCa da MEDIANA das diferenças pareadas\n")

# Mantendo a mesma lógica matemática do seu código:
# IC da mediana das diferenças pareadas
def estatistica_diferenca(amostra, axis=0):
    return np.median(amostra, axis=axis)

resultados = []
p_valores = []

for dim in range(n_dimensoes):
    print("\n" + "=" * 90)
    print(f"DIMENSÃO {dim} (S_{dim})")
    print("=" * 90)

    # 1) Extração dos vetores da dimensão atual
    s_antes = antes_arr[:, dim]
    s_depois = depois_arr[:, dim]

    print("1) Vetores extraídos")
    print(f"   before: {np.round(s_antes, 6)}")
    print(f"   after : {np.round(s_depois, 6)}")

    # 2) Estatísticas descritivas
    media_antes = np.mean(s_antes)
    std_antes = np.std(s_antes, ddof=1)   # se quiser desvio amostral, troque para np.std(s_antes, ddof=1)
    media_depois = np.mean(s_depois)
    std_depois = np.std(s_depois, ddof=1) # se quiser desvio amostral, troque para np.std(s_depois, ddof=1)

    print("\n2) Estatísticas descritivas")
    print(f"   Before -> média = {media_antes:.6f}, desvio = {std_antes:.6f}")
    print(f"   After  -> média = {media_depois:.6f}, desvio = {std_depois:.6f}")

    # 3) Diferenças pareadas
    diferencas = s_antes - s_depois
    diff_validas = diferencas[diferencas != 0]

    print("\n3) Diferenças pareadas (before - after)")
    print(f"   diferenças       = {np.round(diferencas, 6)}")
    print(f"   diferenças != 0  = {np.round(diff_validas, 6)}")

    # 4) Wilcoxon + RBC
    if len(diff_validas) == 0:
        estatistica_w = 0.0
        p_valor = 1.0
        rbc = 0.0

        print("\n4) Wilcoxon / RBC")
        print("   Todas as diferenças são zero.")
        print("   Definindo: W = 0.0, p = 1.0, RBC = 0.0")
    else:
        postos = rankdata(np.abs(diff_validas))
        w_mais = np.sum(postos[diff_validas > 0])
        w_menos = np.sum(postos[diff_validas < 0])

        estatistica_w, p_valor = wilcoxon(s_antes, s_depois)

        soma_w = w_mais + w_menos
        rbc = (w_mais - w_menos) / soma_w if soma_w != 0 else 0.0

        print("\n4) Wilcoxon / RBC")
        print(f"   postos |dif|     = {np.round(postos, 6)}")
        print(f"   W+               = {w_mais:.6f}")
        print(f"   W-               = {w_menos:.6f}")
        print(f"   Estatística W    = {estatistica_w}")
        print(f"   Valor-p          = {p_valor:.6f}")
        print(f"   RBC              = {rbc:.6f}")

    # 5) IC 95% via bootstrap da MEDIANA das diferenças
    try:
        resultado_ic = bootstrap(
            (diferencas,),
            estatistica_diferenca,
            confidence_level=0.95,
            method='BCa',
            n_resamples=10000,
            random_state=42
        )
        ic_low = resultado_ic.confidence_interval.low
        ic_high = resultado_ic.confidence_interval.high
        metodo_ic = "BCa"

    except Exception as e:
        print("\n5) Bootstrap BCa falhou; tentando método percentile.")
        print(f"   Motivo: {e}")

        resultado_ic = bootstrap(
            (diferencas,),
            estatistica_diferenca,
            confidence_level=0.95,
            method='percentile',
            n_resamples=10000,
            random_state=42
        )
        ic_low = resultado_ic.confidence_interval.low
        ic_high = resultado_ic.confidence_interval.high
        metodo_ic = "percentile"

    print("\n5) Intervalo de confiança (95%) da mediana das diferenças")
    print(f"   Método           = {metodo_ic}")
    print(f"   IC95%            = [{ic_low:.6f}, {ic_high:.6f}]")

    resultados.append({
        "Dimensão": f"S_{dim}",
        "Before (Mean ± STD)": f"{media_antes:.4f} ± {std_antes:.4f}",
        "After (Mean ± STD)": f"{media_depois:.4f} ± {std_depois:.4f}",
        "Wilcoxon W": float(estatistica_w),
        "p-value": float(p_valor),
        "RBC": float(rbc),
        "IC95% mediana diff": f"[{ic_low:.4f}, {ic_high:.4f}]"
    })

    p_valores.append(float(p_valor))

# 6) Correção FDR (útil para a resposta aos revisores)
try:
    from statsmodels.stats.multitest import multipletests

    rejeita_h0, p_fdr, _, _ = multipletests(
        p_valores, alpha=0.05, method='fdr_bh'
    )

    for i in range(len(resultados)):
        resultados[i]["p-value FDR"] = float(p_fdr[i])
        resultados[i]["Significativo após FDR?"] = bool(rejeita_h0[i])

    print("\n" + "=" * 90)
    print("CORREÇÃO PARA MÚLTIPLAS COMPARAÇÕES (FDR-BH)")
    print("=" * 90)
    for i, (p_orig, p_corr, rejeita) in enumerate(zip(p_valores, p_fdr, rejeita_h0)):
        print(
            f"S_{i}: p original = {p_orig:.6f} | "
            f"p FDR = {p_corr:.6f} | "
            f"significativo? {rejeita}"
        )

except Exception as e:
    print("\nNão foi possível calcular FDR nesta célula.")
    print(f"Motivo: {e}")

# 7) Tabela final
df_resultados = pd.DataFrame(resultados)

print("\n" + "=" * 90)
print("TABELA FINAL")
print("=" * 90)
print(df_resultados.to_string(index=False))


INÍCIO DA ANÁLISE ESTATÍSTICA
Número de sujeitos   : 9
Número de dimensões  : 4
Shape de 'antes'     : (9, 4)
Shape de 'depois'    : (9, 4)

Observações:
- Diferenças calculadas como: antes - depois
- RBC calculado da mesma forma que você já vinha usando
- IC 95% calculado por bootstrap BCa da MEDIANA das diferenças pareadas


DIMENSÃO 0 (S_0)
1) Vetores extraídos
   before: [4.548446 4.560532 4.529653 4.549154 4.551524 4.50768  4.555997 4.535484
 4.552274]
   after : [4.55463  4.558774 4.506685 4.516198 4.576668 4.544241 4.565908 4.52903
 4.557181]

2) Estatísticas descritivas
   Before -> média = 4.543416, desvio = 0.016519
   After  -> média = 4.545479, desvio = 0.023511

3) Diferenças pareadas (before - after)
   diferenças       = [-0.006184  0.001759  0.022967  0.032956 -0.025144 -0.036561 -0.00991
  0.006453 -0.004907]
   diferenças != 0  = [-0.006184  0.001759  0.022967  0.032956 -0.025144 -0.036561 -0.00991
  0.006453 -0.004907]

4) Wilcoxon / RBC
   postos |dif|     = [3. 1. 